# Text Dataset Cleaning
### MedAI Diagnose | CNN + NLP + PEPA
**Run order:** Step 1 — run this first before any training

**Datasets cleaned:**
- DDXPlus (figshare — 1.3M patients)
- itachi9604 (Kaggle — 41 diseases, severity weights)
- uom190346a (Kaggle — patient profiles with age/sex)
- abhishekgodara (Kaggle — 713 diseases, 377 symptoms)

**Output:** `dataset/cleaned/text/master_text_dataset.csv`

**Columns:** `disease | symptom_text | age | sex`

In [ ]:
# Cell 1 — Imports and Config
import os, re, json, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

# ── PATHS — set to your download locations ───────────────────
BASE = r'C:\Users\Aditya Srivastava\OneDrive\Desktop\MedAI-Diagnose'

DDXPLUS_DIR = BASE + r'\dataset\raw\text\ddxplus\\'
ITACHI_DIR  = BASE + r'\dataset\raw\text\itachi9604\\'
UOM_DIR     = BASE + r'\dataset\raw\text\uom190346a\\'
ABHI_DIR    = BASE + r'\dataset\raw\text\abhishekgodara\\'

OUT_DIR  = BASE + r'\dataset\cleaned\text\\'
OUT_FILE = OUT_DIR + 'master_text_dataset.csv'

# ── Cleaning config ──────────────────────────────────────────
MIN_SAMPLES     = 50    # drop classes with fewer rows
MAX_SAMPLES     = 400   # cap per class to balance
MIN_SYMPTOM_LEN = 3     # drop rows with fewer than 3 words

os.makedirs(OUT_DIR, exist_ok=True)
print('✓ Configuration loaded')

✓ Configuration loaded


In [12]:
# Cell 2 — Helper Functions

def clean_text(t):
    if pd.isna(t): return ''
    return re.sub(r'\s+', ' ', str(t).lower().strip().replace('_', ' '))

def clean_disease(d): return clean_text(d)
def count_words(t): return len(str(t).split())
def banner(title): print(f"\n{'='*55}\n  {title}\n{'='*55}")

print('✓ Helpers ready')

✓ Helpers ready


In [13]:
# Cell 3 — Clean Dataset 1: DDXPlus
banner('DATASET 1 — DDXPlus (1.3M patients, severity levels)')

train_path = os.path.join(DDXPLUS_DIR, 'release_train_patients.csv')
evid_path  = os.path.join(DDXPLUS_DIR, 'release_evidences.json')

if not os.path.exists(train_path):
    print(f'  [SKIP] Not found: {train_path}')
    print('  Download: https://figshare.com/articles/dataset/DDXPlus_Dataset_English_/22687585')
    ds1 = None
else:
    df_ddx = pd.read_csv(train_path)
    print(f'  Loaded: {df_ddx.shape}')

    ev_names = {}
    if os.path.exists(evid_path):
        with open(evid_path) as f:
            for code, info in json.load(f).items():
                name = info.get('name', code)
                if isinstance(name, dict): name = name.get('en', code)
                ev_names[code] = clean_text(name)

    def parse_ev(ev_str):
        try:
            items = ev_str.strip('[]').replace("'", '').split(',')
            syms  = [ev_names.get(i.strip().split('_@_')[0], clean_text(i.strip().split('_@_')[0]))
                     for i in items if i.strip()]
            return ' '.join(s for s in syms if s)
        except: return ''

    df_ddx['symptom_text'] = df_ddx['EVIDENCES'].apply(parse_ev)
    df_ddx['disease']      = df_ddx['PATHOLOGY'].apply(clean_disease)
    df_ddx['age']          = pd.to_numeric(df_ddx.get('AGE', 0), errors='coerce').fillna(0).astype(int)
    df_ddx['sex']          = df_ddx.get('SEX', 'U').fillna('U').astype(str).str.upper().str[0]

    ds1 = df_ddx[['disease', 'symptom_text', 'age', 'sex']]
    ds1 = ds1[ds1['symptom_text'].str.strip() != '']
    print(f'  ✓ {len(ds1):,} rows  |  {ds1["disease"].nunique()} diseases')
    print(ds1.head(2).to_string())


  DATASET 1 — DDXPlus (1.3M patients, severity levels)
  Loaded: (1025602, 6)
  ✓ 1,025,602 rows  |  49 diseases
                   disease                                                                                                                                                                symptom_text  age sex
0                     urti                                                                           e 48 e 50 e 53 e 54 e 54 e 55 e 55 e 55 e 56 e 57 e 58 e 59 e 77 e 79 e 91 e 97 e 201 e 204 e 222   18   M
1  hiv (initial infection)  e 9 e 27 e 50 e 51 e 53 e 54 e 55 e 55 e 55 e 56 e 57 e 58 e 59 e 91 e 115 e 129 e 130 e 131 e 132 e 133 e 133 e 133 e 133 e 133 e 134 e 135 e 136 e 148 e 162 e 189 e 204   21   M


In [14]:
# Cell 4 — Clean Dataset 2: itachi9604
banner('DATASET 2 — itachi9604 (41 diseases, severity weights)')

data_path = None
for fname in ['dataset.csv', 'Dataset.csv', 'training.csv']:
    p = os.path.join(ITACHI_DIR, fname)
    if os.path.exists(p): data_path = p; break

if data_path is None:
    print(f'  [SKIP] Not found in: {ITACHI_DIR}')
    print('  Download: https://www.kaggle.com/datasets/itachi9604/disease-symptom-description-dataset')
    ds2 = None
else:
    df_it = pd.read_csv(data_path)
    for col in df_it.columns:
        if df_it[col].dtype == object:
            df_it[col] = df_it[col].str.replace('_', ' ').str.strip()

    sev_weights = {}
    sev_path = os.path.join(ITACHI_DIR, 'Symptom-severity.csv')
    if os.path.exists(sev_path):
        sv = pd.read_csv(sev_path)
        sv.columns = sv.columns.str.strip().str.lower()
        sc = [c for c in sv.columns if 'symptom' in c][0]
        wc = [c for c in sv.columns if 'weight' in c][0]
        sv[sc] = sv[sc].str.replace('_', ' ').str.strip().str.lower()
        sev_weights = dict(zip(sv[sc], sv[wc]))

    dis_col  = [c for c in df_it.columns if c.lower() == 'disease'][0]
    sym_cols = [c for c in df_it.columns if c.lower().startswith('symptom')]

    rows = []
    for _, row in df_it.iterrows():
        disease  = clean_disease(row[dis_col])
        symptoms = []
        for col in sym_cols:
            val = row[col]
            if pd.notna(val) and str(val).strip():
                sym = clean_text(str(val))
                symptoms.extend([sym] * max(1, int(sev_weights.get(sym, 1) / 2)))
        if symptoms:
            rows.append({'disease': disease, 'symptom_text': ' '.join(symptoms), 'age': 0, 'sex': 'U'})

    ds2 = pd.DataFrame(rows)
    print(f'  ✓ {len(ds2):,} rows  |  {ds2["disease"].nunique()} diseases')
    print(ds2.head(2).to_string())


  DATASET 2 — itachi9604 (41 diseases, severity weights)
  ✓ 4,920 rows  |  41 diseases
            disease                                                                                                          symptom_text  age sex
0  fungal infection  itching skin rash nodal skin eruptions nodal skin eruptions dischromic patches dischromic patches dischromic patches    0   U
1  fungal infection          skin rash nodal skin eruptions nodal skin eruptions dischromic patches dischromic patches dischromic patches    0   U


In [15]:
# Cell 5 — Clean Dataset 3: uom190346a
banner('DATASET 3 — uom190346a (patient profile + demographics)')

data_path = None
for fname in ['Disease_symptom_and_patient_profile.csv', 'disease_symptom.csv', 'dataset.csv']:
    p = os.path.join(UOM_DIR, fname)
    if os.path.exists(p): data_path = p; break

if data_path is None:
    print(f'  [SKIP] Not found in: {UOM_DIR}')
    print('  Download: https://www.kaggle.com/datasets/uom190346a/disease-symptoms-and-patient-profile-dataset')
    ds3 = None
else:
    df_uom = pd.read_csv(data_path)
    print(f'  Loaded: {df_uom.shape}  Columns: {df_uom.columns.tolist()}')

    dis_col    = next((c for c in df_uom.columns if c.lower() == 'disease'), df_uom.columns[0])
    age_col    = next((c for c in df_uom.columns if 'age' in c.lower()), None)
    gender_col = next((c for c in df_uom.columns if 'gender' in c.lower() or 'sex' in c.lower()), None)
    bp_col     = next((c for c in df_uom.columns if 'blood' in c.lower()), None)
    chol_col   = next((c for c in df_uom.columns if 'cholesterol' in c.lower()), None)

    binary_cols = [c for c in df_uom.columns
                   if set(df_uom[c].dropna().astype(str).str.lower().unique())
                      .issubset({'yes','no','1','0'})
                   and c.lower() not in ['outcome variable','outcome']]

    rows = []
    for _, row in df_uom.iterrows():
        disease  = clean_disease(row[dis_col])
        symptoms = [clean_text(c) for c in binary_cols
                    if str(row.get(c, 'no')).strip().lower() in ['yes','1']]
        if bp_col and str(row.get(bp_col,'')).strip().lower() in ['low','high']:
            symptoms.append(f"{str(row[bp_col]).lower()} blood pressure")
        if chol_col and str(row.get(chol_col,'')).strip().lower() in ['low','high']:
            symptoms.append(f"{str(row[chol_col]).lower()} cholesterol")
        if not symptoms: continue
        age = 0
        if age_col:
            try: age = int(float(row[age_col]))
            except: pass
        sex = 'U'
        if gender_col:
            g = str(row.get(gender_col,'')).strip().lower()
            sex = 'M' if g in ['male','m'] else ('F' if g in ['female','f'] else 'U')
        rows.append({'disease': disease, 'symptom_text': ' '.join(symptoms), 'age': age, 'sex': sex})

    ds3 = pd.DataFrame(rows)
    print(f'  ✓ {len(ds3):,} rows  |  {ds3["disease"].nunique()} diseases')


  DATASET 3 — uom190346a (patient profile + demographics)
  Loaded: (349, 10)  Columns: ['Disease', 'Fever', 'Cough', 'Fatigue', 'Difficulty Breathing', 'Age', 'Gender', 'Blood Pressure', 'Cholesterol Level', 'Outcome Variable']
  ✓ 348 rows  |  116 diseases


In [19]:
# Cell 6 — Clean Dataset 4: abhishekgodara
banner('DATASET 4 — abhishekgodara (713 diseases, 377 symptoms)')

proc_path = os.path.join(ABHI_DIR, 'processed_dataset.csv')
raw_path  = os.path.join(ABHI_DIR, 'raw_dataset.csv')

used_path = None
for path in [proc_path, raw_path]:
    if os.path.exists(path):
        used_path = path
        break

if used_path is None:
    print(f'  [SKIP] Not found in: {ABHI_DIR}')
    ds4 = None
else:
    df_ab = pd.read_csv(used_path)
    print(f'  Loaded: {df_ab.shape}')
    print(f'  Columns: {df_ab.columns.tolist()}')

    # Find disease column — handles 'disease', 'diseases', 'prognosis' etc
    dis_col = next(
        (c for c in df_ab.columns
         if c.lower() in ['disease', 'diseases', 'prognosis', 'label', 'target']),
        df_ab.columns[0]
    )
    print(f'  Disease column: {dis_col}')

    # Find symptom text column
    txt_col = next(
        (c for c in df_ab.columns
         if c.lower() in ['symptom_text', 'text', 'symptoms', 'description']),
        None
    )
    print(f'  Text column: {txt_col}')

    rows = []

    if txt_col is not None:
        # ── PATH A: already has symptom_text column ──────────────
        # Your file has exactly 2 cols: diseases + symptom_text
        print('  Mode: NLP text column detected')
        for _, row in df_ab.iterrows():
            d = clean_disease(row[dis_col])
            t = clean_text(str(row[txt_col]))
            if d and t:
                rows.append({
                    'disease':      d,
                    'symptom_text': t,
                    'age':          0,
                    'sex':          'U'
                })
    else:
        # ── PATH B: binary matrix (0/1 columns) ──────────────────
        print('  Mode: binary matrix')
        sym_cols = [c for c in df_ab.columns if c != dis_col]
        for _, row in df_ab.iterrows():
            d    = clean_disease(row[dis_col])
            syms = []
            for col in sym_cols:
                try:
                    if float(row[col]) == 1:
                        syms.append(clean_text(col))
                except Exception:
                    if pd.notna(row[col]) and str(row[col]).strip():
                        syms.append(clean_text(str(row[col])))
            if d and syms:
                rows.append({
                    'disease':      d,
                    'symptom_text': ' '.join(syms),
                    'age':          0,
                    'sex':          'U'
                })

    ds4 = pd.DataFrame(rows)
    print(f'  ✓ {len(ds4):,} rows  |  {ds4["disease"].nunique()} diseases')
    print(ds4.head(2).to_string())


  DATASET 4 — abhishekgodara (713 diseases, 377 symptoms)
  Loaded: (192715, 2)
  Columns: ['diseases', 'symptom_text']
  Disease column: diseases
  Text column: symptom_text
  Mode: NLP text column detected
  ✓ 192,715 rows  |  254 diseases
          disease                                                                                                                                        symptom_text  age sex
0  panic disorder  anxiety and nervousness, shortness of breath, depressive or psychotic symptoms, chest tightness, palpitations, irregular heartbeat, breathing fast    0   U
1  panic disorder                                                            shortness of breath, depressive or psychotic symptoms, dizziness, insomnia, palpitations    0   U


In [21]:
# Cell 7 — Merge and Final Cleaning
banner('MERGING AND CLEANING')

valid_dfs = [df for df in [ds1, ds2, ds3, ds4] if df is not None and len(df) > 0]
print(f'  Merging {len(valid_dfs)} datasets...')
for i, df in enumerate(valid_dfs):
    print(f'    ds{i+1}: {len(df):,} rows, {df["disease"].nunique()} diseases')

combined = pd.concat(valid_dfs, ignore_index=True)

# Standardise disease names
combined['disease'] = (combined['disease'].str.lower().str.strip()
                       .str.replace(r'\s+', ' ', regex=True)
                       .str.replace(r'[^\w\s]', '', regex=True))

# Remove short/empty rows
combined = combined[combined['symptom_text'].str.strip() != '']
combined = combined[combined['symptom_text'].apply(count_words) >= MIN_SYMPTOM_LEN]
combined = combined[combined['disease'].str.strip() != '']

# Remove exact duplicates
before = len(combined)
combined = combined.drop_duplicates(subset=['disease', 'symptom_text']).reset_index(drop=True)
print(f'  Removed {before - len(combined):,} duplicates')

# Filter rare classes
counts  = combined['disease'].value_counts()
valid   = counts[counts >= MIN_SAMPLES].index
combined = combined[combined['disease'].isin(valid)].reset_index(drop=True)
print(f'  After rare filter: {combined["disease"].nunique()} diseases')

# Balance classes — pandas 2.x compatible
np.random.seed(42)
balanced_parts = []
for disease_name, group in combined.groupby('disease'):
    n = min(len(group), MAX_SAMPLES)
    balanced_parts.append(group.sample(n=n, random_state=42))

combined = pd.concat(balanced_parts, ignore_index=True)
combined = combined.reset_index(drop=True)

cc = combined['disease'].value_counts()
print(f'\n  FINAL: {len(combined):,} rows  |  {combined["disease"].nunique()} diseases')
print(f'  Samples: min={cc.min()} max={cc.max()} mean={cc.mean():.0f}')

combined.to_csv(OUT_FILE, index=False)
print(f'\n  ✓ SAVED → {OUT_FILE}')
combined.head()


  MERGING AND CLEANING
  Merging 4 datasets...
    ds1: 1,025,602 rows, 49 diseases
    ds2: 4,920 rows, 41 diseases
    ds3: 348 rows, 116 diseases
    ds4: 192,715 rows, 254 diseases
  Removed 747,227 duplicates
  After rare filter: 297 diseases

  FINAL: 110,788 rows  |  297 diseases
  Samples: min=79 max=400 mean=373

  ✓ SAVED → C:\Users\Aditya Srivastava\OneDrive\Desktop\MedAI-Diagnose\dataset\cleaned\text\\master_text_dataset.csv


,disease,symptom_text,age,sex
0,abdominal hernia,"symptoms of the scrotum and testes, lower abdo...",0,U
1,abdominal hernia,"symptoms of the scrotum and testes, regurgitat...",0,U
2,abdominal hernia,"symptoms of the scrotum and testes, regurgitat...",0,U
3,abdominal hernia,"sharp abdominal pain, regurgitation, upper abd...",0,U
4,abdominal hernia,"groin mass, symptoms of the scrotum and testes...",0,U
